# Artmatcher Feature Demo

This notebook demonstrates the two image-level feature groups used by the artmatcher classifier:

- **Perceptual hashes** — compact fingerprints that stay similar when the image changes slightly
- **ORB features** — local keypoints that enable geometric matching between two images

Both are computed on raw pixels (no model needed) and complement the embedding-based distribution features.

# Setup

In [ ]:
import io
import urllib.request

import cv2
import imagehash
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from artmatcher.features import distribution_features, make_orb

Helper to download an image from a URL (adds `User-Agent` header required by some hosts):

In [ ]:
def fetch_image(url: str) -> Image.Image:
    req = urllib.request.Request(url, headers={"User-Agent": "glashelder-notebook/1.0"})
    with urllib.request.urlopen(req) as resp:
        return Image.open(io.BytesIO(resp.read())).convert("RGB")

---

# Part 1: Perceptual Hashes

Perceptual hashes convert an image into a compact binary fingerprint. Unlike cryptographic hashes (MD5, SHA), they are designed so that **visually similar images produce similar hashes** — small changes to the image → small change in the hash.

Similarity between two hashes is measured by their [Hamming distance](https://en.wikipedia.org/wiki/Hamming_distance): the number of bit positions that differ. A distance of `0` means identical; a distance of `64` (for a 64-bit hash) means completely different.

This section demonstrates four hash types on a real image and shows how each reacts to a simple modification (adding a black rectangle).

In [ ]:
# Modify IMAGE_URL to any publicly accessible image URL
IMAGE_URL = "https://sipi.usc.edu/database/download.php?vol=misc&img=4.1.05"

img_original = fetch_image(IMAGE_URL)
print(f"Downloaded image: {img_original.size[0]}x{img_original.size[1]} px")
img_original

## 1.1 Hash Types

The artmatcher uses four perceptual hash algorithms, each looking at the image slightly differently:

| Hash | What it looks at |
|------|-----------------|
| **pHash** (perceptual) | Overall visual "shape" of the image, downscaled |
| **dHash** (difference) | Whether each pixel is lighter or darker than its neighbour |
| **wHash** (wavelet) | Visual patterns at multiple zoom levels simultaneously |
| **aHash** (average) | Whether each region is above or below the image's average brightness |

All produce 64-bit hashes by default. The artmatcher uses all four as separate features so the classifier can weigh each independently.

In [ ]:
def compute_hashes(img: Image.Image) -> dict:
    return {
        "pHash": imagehash.phash(img),
        "dHash": imagehash.dhash(img),
        "wHash": imagehash.whash(img),
        "aHash": imagehash.average_hash(img),
    }


hashes_original = compute_hashes(img_original)

print("Hashes for original image:")
for name, h in hashes_original.items():
    print(f"  {name}: {h}")

## 1.2 Effect of a Small Modification

We apply three progressively larger black rectangles to the image and measure how the hash distances change. The key observation: hashes change only slightly for small modifications — the distance grows gradually as more of the image is obscured.

In [ ]:
def add_black_region(img: Image.Image, fraction: float) -> Image.Image:
    """Cover the top-left corner with a black rectangle (fraction = share of width/height)."""
    arr = np.array(img).copy()
    h, w = arr.shape[:2]
    bh, bw = int(h * fraction), int(w * fraction)
    arr[:bh, :bw] = 0
    return Image.fromarray(arr)


def resize(img: Image.Image, fraction: float) -> Image.Image:
    return img.resize((round(img.width * fraction), round(img.height * fraction)))


def rotate_and_zoom(img: Image.Image, angle: float) -> Image.Image:
    return img.rotate(angle, resample=Image.BICUBIC, expand=False)  # ty:ignore[unresolved-attribute]


modifications = {
    "10% corner blacked": add_black_region(img_original, 0.10),
    "25% corner blacked": add_black_region(img_original, 0.25),
    "50% corner blacked": add_black_region(img_original, 0.50),
    "50% size": resize(img_original, 0.50),
    "10% size": resize(img_original, 0.10),
    "10deg rotation": rotate_and_zoom(img_original, 10),
}

print(f"{'Modification':<25} {'pHash':>6} {'dHash':>6} {'wHash':>6} {'aHash':>6}")
print("-" * 55)
for label, img_mod in modifications.items():
    hashes_mod = compute_hashes(img_mod)
    dists = {k: hashes_original[k] - hashes_mod[k] for k in hashes_original}
    print(
        f"{label:<25} {dists['pHash']:>6} {dists['dHash']:>6} {dists['wHash']:>6} {dists['aHash']:>6}"
    )

print("\n(values = Hamming distance; max = 64; 0 = identical)")

In [ ]:
fig, axes = plt.subplots(1, len(modifications) + 1, figsize=(18, 5))

images = {"Original": img_original, **modifications}
for ax, (label, img) in zip(axes, images.items(), strict=True):
    ax.imshow(img)
    ax.set_title(label)
    ax.axis("off")

plt.suptitle("Original vs. modified images", y=1.02)
plt.tight_layout()
plt.show()

---

# Part 2: ORB Features

[ORB (Oriented FAST and Rotated BRIEF)](https://docs.opencv.org/4.x/d1/d89/tutorial_py_orb.html) detects distinctive local regions in an image called **keypoints** — corners, blobs, edges — and describes each one with a compact binary descriptor. The descriptor is designed to be invariant to rotation and scale changes, so the same physical region in two different photos of the same scene produces similar descriptors.

The artmatcher uses ORB to answer the question: *do these two images depict the same physical object/scene?* If many keypoints from image A match keypoints in image B (and those matches are geometrically consistent), that's strong evidence of a true match.

The ORB pipeline:
1. Detect keypoints in both images
2. Describe each keypoint with a 256-bit BRIEF descriptor
3. Match descriptors using Hamming distance + Lowe's ratio test (filters weak matches)
4. Estimate a homography (projective transform) with RANSAC to count geometrically consistent **inliers**

The artmatcher extracts five ORB features from this pipeline:
- `orb_num_matches` — total good matches after ratio test
- `orb_num_inliers` — matches consistent with the estimated homography
- `orb_confidence` — inlier ratio (inliers / total keypoints)
- `orb_warp_stretch` — how much the homography stretches/distorts the image (large stretch = suspicious)
- `inlier_match_ratio` — inliers / total matches

## 2.1 Keypoint Detection

Detect and visualize ORB keypoints on the image downloaded above.

In [ ]:
# Convert PIL → BGR for OpenCV
img_bgr = cv2.cvtColor(np.array(img_original), cv2.COLOR_RGB2BGR)
img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

orb = cv2.ORB_create(nfeatures=500)  # ty:ignore[unresolved-attribute]
keypoints, descriptors = orb.detectAndCompute(img_gray, None)

print(f"Detected {len(keypoints)} keypoints")

img_kp = cv2.drawKeypoints(img_bgr, keypoints, None, color=(0, 255, 0))  # ty:ignore[no-matching-overload]

plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(img_kp, cv2.COLOR_BGR2RGB))
plt.title(f"ORB keypoints ({len(keypoints)} detected)")
plt.axis("off")
plt.show()

Each green dot is a detected keypoint — a distinctive local region (corner, blob, edge junction) that ORB can reliably re-identify in a transformed version of the same image. Keypoints cluster around high-contrast areas and edges, avoiding flat/uniform regions which carry little distinctive information.

## 2.2 Matching Between Similar Images

We create a slightly modified copy of the image (rotated 5°, slightly zoomed) and match keypoints between the two. Since they depict the same scene, most keypoints should match.

In [ ]:
def rotate_and_zoom(
    img: Image.Image, angle: float = 5.0, zoom: float = 1.05
) -> Image.Image:
    """Rotate by `angle` degrees and zoom in slightly."""
    w, h = img.size
    # Zoom by cropping a centered region, then resizing back
    crop_w, crop_h = int(w / zoom), int(h / zoom)
    left = (w - crop_w) // 2
    top = (h - crop_h) // 2
    img_zoomed = img.crop((left, top, left + crop_w, top + crop_h)).resize(
        (w, h), Image.LANCZOS
    )  # ty:ignore[unresolved-attribute]
    return img_zoomed.rotate(angle, resample=Image.BICUBIC, expand=False)  # ty:ignore[unresolved-attribute]


img_modified = rotate_and_zoom(img_original, angle=8.0, zoom=1.1)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(img_original)
axes[0].set_title("Original")
axes[0].axis("off")
axes[1].imshow(img_modified)
axes[1].set_title("Modified (rotated 8°, zoomed 1.1x)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
def to_bgr(img: Image.Image):
    return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)


def detect_and_match(img_a: Image.Image, img_b: Image.Image, n_features: int = 500):
    """Detect ORB keypoints in both images and return good matches after ratio test."""
    orb = cv2.ORB_create(nfeatures=n_features)  # ty:ignore[unresolved-attribute]
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)

    bgr_a = to_bgr(img_a)
    bgr_b = to_bgr(img_b)
    gray_a = cv2.cvtColor(bgr_a, cv2.COLOR_BGR2GRAY)
    gray_b = cv2.cvtColor(bgr_b, cv2.COLOR_BGR2GRAY)

    kp_a, des_a = orb.detectAndCompute(gray_a, None)
    kp_b, des_b = orb.detectAndCompute(gray_b, None)

    raw_matches = bf.knnMatch(des_a, des_b, k=2)
    good = [m for m, n in raw_matches if m.distance < 0.75 * n.distance]

    return kp_a, kp_b, good, bgr_a, bgr_b


kp_orig, kp_mod, good_matches, bgr_orig, bgr_mod = detect_and_match(
    img_original, img_modified
)
print(f"Good matches (original ↔ modified): {len(good_matches)}")

# Draw top 40 matches
match_img = cv2.drawMatches(
    bgr_orig,
    kp_orig,
    bgr_mod,
    kp_mod,
    good_matches[:40],
    None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
)  # ty:ignore[no-matching-overload]

plt.figure(figsize=(18, 7))
plt.imshow(cv2.cvtColor(match_img, cv2.COLOR_BGR2RGB))
plt.title(f"Top 40 of {len(good_matches)} matches — original vs. modified")
plt.axis("off")
plt.show()

Lines connect matched keypoints across the two images. A high number of consistent matches (lines that don't cross wildly) indicates the two images show the same scene.

## 2.3 Comparison: Similar vs. Different Image

To show the contrast, we download a second, unrelated image and match it against the original. Expect far fewer (or zero) good matches.

In [ ]:
# Any unrelated public-domain image
UNRELATED_URL = "https://sipi.usc.edu/database/download.php?vol=misc&img=4.2.01"

img_unrelated = fetch_image(UNRELATED_URL)

kp_orig2, kp_unrel, matches_unrel, bgr_orig2, bgr_unrel = detect_and_match(
    img_original, img_unrelated
)
print(f"Good matches (original ↔ unrelated): {len(matches_unrel)}")
print(f"Good matches (original ↔ modified): {len(good_matches)}")

if matches_unrel:
    match_img2 = cv2.drawMatches(
        bgr_orig2,
        kp_orig2,
        bgr_unrel,
        kp_unrel,
        matches_unrel[:40],
        None,
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
    )  # ty:ignore[no-matching-overload]
    plt.figure(figsize=(18, 7))
    plt.imshow(cv2.cvtColor(match_img2, cv2.COLOR_BGR2RGB))
    plt.title(f"Top 40 of {len(matches_unrel)} matches — original vs. unrelated")
    plt.axis("off")
    plt.show()
else:
    print("No good matches found — as expected for unrelated images.")

## 2.4 Extracting the ORB Feature Vector

`FeatureExtractor.extract_orb_features()` runs the full ORB pipeline (including RANSAC homography) and returns the five features the classifier uses. It works on file paths, so we save our images temporarily.

In [ ]:
import tempfile
from pathlib import Path

extractor = FeatureExtractor()

with tempfile.TemporaryDirectory() as tmp:
    path_orig = str(Path(tmp) / "original.jpg")
    path_mod = str(Path(tmp) / "modified.jpg")
    path_unrel = str(Path(tmp) / "unrelated.jpg")

    img_original.save(path_orig)
    img_modified.save(path_mod)
    img_unrelated.save(path_unrel)

    feats_similar = extractor.extract_orb_features(path_orig, path_mod)
    feats_different = extractor.extract_orb_features(path_orig, path_unrel)

print("ORB features — similar (original vs. modified):")
print(f"  num_matches:      {feats_similar.num_matches}")
print(f"  num_inliers:      {feats_similar.num_inliers}")
print(f"  confidence:       {feats_similar.confidence:.4f}")
print(f"  warp_stretch:     {feats_similar.warp_stretch:.4f}")

print("\nORB features — different (original vs. unrelated):")
print(f"  num_matches:      {feats_different.num_matches}")
print(f"  num_inliers:      {feats_different.num_inliers}")
print(f"  confidence:       {feats_different.confidence:.4f}")
print(f"  warp_stretch:     {feats_different.warp_stretch:.4f}")

The similar pair should show high `num_inliers` and `confidence`; the unrelated pair should show near-zero values. These five numbers are what the Random Forest sees — along with the hash distances and distribution features — when deciding whether two images are a match.